## 5.3 跳频分集仿真

在上一节中，我们学习了跳频和 Rayleigh 衰落的原理。本节通过仿真对比**跳频**（每帧独立信道实现）与**固定信道**（所有帧共用一个信道实现）在 Rayleigh 衰落下的 FER，直观展示跳频分集增益。

> **仿真简化说明**：本仿真不涉及载波频率映射和 PRNG 信道号计算（`hopping_prng` / `data_link_hop`），而是直接给每帧分配独立的 Rayleigh 随机种子，使信道系数独立变化。这是统计等效做法——真正的跳频通过切换载波频率改变多径环境，仿真中用独立随机信道系数模拟同一效果。优点是代码聚焦分集增益的核心机制，缺点是无法展示阻塞信道管理、带宽约束等标准协议层细节。

本节学习大纲如下：

- Rayleigh 衰落信道建模与分集原理
- 跳频 vs 固定信道 FER 扫描
- 多实现统计与标准差分析

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── phy/
│   └── channel.py         <- ChannelModel / ChannelConfig: Rayleigh 信道建模
├── common/
│   └── polar.py           <- PolarEncoder / get_polar_decoder: Polar 编译码
└── sim/
    └── link_sim.py        <- sim_hopping_multipath_link: 跳频 vs 固定信道仿真
```

---

### 1. Rayleigh 衰落与分集原理

AWGN 假设信号只受热噪声影响。但在多径环境中，信号经过不同路径到达接收端会产生干涉，导致信号强度剧烈波动——这就是**衰落**。

瑞利衰落（Rayleigh Fading）：在无线通信信道中，由于信号进行多径传播达到接收点处的场强来自不同传播的路径，各条路径延时时间是不同的，而各个方向分量波的叠加，又产生了驻波场强，从而形成信号快衰落称为瑞利衰落。瑞利衰落属于小尺度的衰落效应，它总是叠加于如阴影、衰减等大尺度衰落效应上。


Rayleigh 衰落模型适用于无直射路径的密集多径场景。信道系数 $h$ 服从复高斯分布 $h \sim \mathcal{CN}(0, 1)$，接收信号为：

$$r = h \cdot s + n$$

其中 $|h|$ 服从 Rayleigh 分布。$|h|$ 很小 → 深衰落 → 该帧几乎必然出错；$|h|$ 大 → 信号强 → 帧容易正确。


Rayleigh 衰落信道的相关函数如下：

**ChannelConfig** — 信道配置参数，`channel_type` 支持 `"awgn"` / `"rayleigh"` / `"rician"` / `"multipath"`，`seed` 控制随机种子：

```python
class ChannelConfig:
    snr_db: float = 10.0
    channel_type: str = "awgn"          # "awgn" | "rayleigh" | "rician" | "multipath"
    seed: int | None = None             # 随机种子, 不同种子→不同信道实现
```

**ChannelModel.apply_fading()** — 按 `channel_type` 应用衰落 + AWGN。Rayleigh 路径生成复高斯系数 `h`，逐元素乘到信号上（$r = h \cdot s + n$）：

```python
def apply_fading(self, signal, sps=1):
    ct = self.cfg.channel_type
    if ct == "rayleigh":
        h = self._gen_rayleigh_coeffs(len(signal))   # h ~ CN(0, 1)
        self._last_taps = h.reshape(1, -1)
        return self._add_noise(signal * h, self.cfg.snr_db, sps)
```

**_gen_rayleigh_coeffs(n)** — 生成 $n$ 个 Rayleigh 衰落系数。无 Doppler（`max_doppler_hz=0`）时为准静态——整段信号共用同一个 $h \sim \mathcal{CN}(0, 1)$（即 `standard_normal() + 1j·standard_normal()` 除以 $\sqrt{2}$）；有 Doppler 时改用 Jakes 求正弦和模型产生时间相关衰落序列，本实验采用准静态分支。
```python
    def _gen_rayleigh_coeffs(self, n: int) -> np.ndarray:
        """生成 Rayleigh 衰落系数。

        max_doppler_hz > 0 时使用 Jakes 求和正弦模型, 产生具有经典 U 形
        功率谱密度的时间相关衰落; 否则为准静态 (整段同一系数)。
        """
        if self.cfg.max_doppler_hz <= 0:
            h = (self._rng.standard_normal() + 1j * self._rng.standard_normal()) / np.sqrt(2)
            return np.full(n, h)
        return self._jakes_fading(n)
```

固定信道一旦信道处于深衰落，后续所有帧都会持续受影响直到信道条件变化。**跳频的解决方案**：每帧换到不同频率 → 多径环境（反射路径长度、相位）完全改变 → 信道系数 $h$ 独立变化 → 深衰落仅影响个别帧，不会持续中断。仿真中的用法：`ChannelModel(config=ChannelConfig(channel_type="rayleigh", seed=...))`，每帧换一个 `seed` 就得到独立的信道实现，给每帧生成独立的 Rayleigh 信道系数。只要频率间隔 ≥ 相干带宽，不同频率的信道就是独立的，所以统计上完全等效，等价于跳频到不同频率。

---

### 2. 跳频 vs 固定信道 FER 对比

Polar(256, 112)，每 SNR 发 100 帧。固定信道取 20 个不同随机实现平均（带标准差）。

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder
from nearlink_sdr.phy.channel import ChannelConfig, ChannelModel

# ===== 仿真参数 =====
snr_range = np.arange(0, 20, 2)   # SNR 扫描范围 0~18 dB
n_frames = 100                     # 每个 SNR 点发送的帧数
n_drops = 20                       # 固定信道的随机实现次数
code_n, K = 256, 112               # Polar 码长 N=256, 信息位 K=112 (码率≈1/2)

# ===== 初始化 =====
rng = np.random.default_rng(42)
enc = PolarEncoder(code_n, K)                    # Polar 编码器
dec = get_polar_decoder(code_n, K)               # Polar SC 译码器

fer_hop = []                                     # 跳频 FER 列表
fer_fix_all = [[] for _ in range(n_drops)]       # 固定信道各次实现的 FER

print(f"{'SNR':>5s}  {'Hop':>8s}  {'FixAvg':>8s}")
print("-" * 28)

for snr in snr_range:
    err_hop = 0                                   # 跳频错误帧数
    snr_lin = 10.0 ** (float(snr) / 10.0)         # SNR dB → 线性值

    for drop in range(n_drops):
        err_fix = 0                                # 本次固定信道实现的错误帧数

        # ----- 生成固定信道的 h（本 drop 内所有帧共用）-----
        ch_tmpl = ChannelModel(config=ChannelConfig(
            snr_db=10.0, channel_type="rayleigh",
            seed=int(rng.integers(0, 2**31))))     # 随机种子 → 独立的 h
        h_fixed = ch_tmpl.get_channel_taps(code_n)[0, :]  # 取出平坦衰落系数 h

        for _ in range(n_frames):
            # ----- 发送端：随机信息比特 → Polar 编码 → BPSK 调制 -----
            info = rng.integers(0, 2, size=K, dtype=np.int8)     # K 个随机信息比特
            coded = enc.encode(info)                              # Polar 编码
            tx = (1 - 2 * coded.astype(np.float64)).astype(complex)  # BPSK: 0→+1, 1→-1

            # ----- AWGN 噪声（噪声方差由 SNR 决定）-----
            noise_std = np.sqrt(1.0 / (2.0 * snr_lin))
            n = noise_std * (rng.standard_normal(code_n)           # 复高斯噪声 I 路
                           + 1j * rng.standard_normal(code_n))    # 复高斯噪声 Q 路

            # ----- 跳频信道：每帧独立 h（只在第一个 drop 算一次）-----
            if drop == 0:
                ch = ChannelModel(config=ChannelConfig(
                    snr_db=float(snr), channel_type="rayleigh",
                    seed=int(rng.integers(0, 2**31))))   # 新随机种子 → 新 h
                h_hop = ch.get_channel_taps(code_n)[0, :]
                rx = tx * h_hop + n                       # r = h·s + n
                # MMSE 均衡 + LLR 计算
                llr = 4 * np.real(rx * np.conj(h_hop)) * snr_lin
                if np.sum(dec.decode(llr) != info) > 0:   # 译码后任一位错误 → 帧错误
                    err_hop += 1

            # ----- 固定信道：所有帧共用 h_fixed -----
            rx = tx * h_fixed + n                          # r = h_fixed·s + n
            llr = 4 * np.real(rx * np.conj(h_fixed)) * snr_lin
            if np.sum(dec.decode(llr) != info) > 0:
                err_fix += 1

        fer_fix_all[drop].append(err_fix / n_frames)       # 本次实现该 SNR 的 FER

    # ----- 统计该 SNR 下的固定信道平均 FER -----
    fer_fix_snr = [fer_fix_all[d][-1] for d in range(n_drops)]
    fer_fix_avg = np.mean(fer_fix_snr)

    fer_hop.append(err_hop / n_frames)                      # 跳频在该 SNR 的 FER
    print(f"{snr:5.0f}  {fer_hop[-1]:8.4f}  {fer_fix_avg:8.4f}")

---

### 3. FER 对比曲线

In [ ]:
import matplotlib.pyplot as plt

fer_fix_avg = [np.mean([fer_fix_all[d][i] for d in range(n_drops)]) for i in range(len(snr_range))]

fig, ax = plt.subplots(figsize=(10, 6))

# 固定信道：每个 SNR 处画出 20 个 drop 的散点
# FER>0 的点正常画在对数坐标上，FER=0 的点画在底边横轴上
for i, snr in enumerate(snr_range):
    fers = np.array([fer_fix_all[d][i] for d in range(n_drops)])
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(fers))
    # 非零点
    nonzero = fers > 0
    if np.any(nonzero):
        ax.scatter(np.full(np.sum(nonzero), snr) + jitter[nonzero],
                   fers[nonzero], c="red", s=25, alpha=0.6,
                   edgecolors="none", zorder=3)
    # 零点画在底边 
    zero = fers == 0
    if np.any(zero):
        ax.scatter(np.full(np.sum(zero), snr) + jitter[zero],
                   np.full(np.sum(zero), 1e-4), c="red", s=25, alpha=0.6,
                   edgecolors="none", zorder=3)

# 固定信道平均值连线
fer_fix_avg_safe = [max(f, 1e-4) for f in fer_fix_avg]
ax.semilogy(snr_range, fer_fix_avg_safe, "o--", color="red",
            lw=1.5, markersize=6, label="Fixed (20 drops)")

# 跳频曲线
fer_hop_safe = [max(f, 1e-4) for f in fer_hop]
ax.semilogy(snr_range, fer_hop_safe, "s-", color="blue",
            lw=2, markersize=8, label="Hopping")

ax.set_xlabel("SNR (dB)"); ax.set_ylabel("FER")
ax.set_title("Hopping vs Fixed Channel")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.6)
ax.set_ylim(bottom=1e-4, top=2)
plt.show()

---

### 4. 实验分析

从上图可以直观得出以下结论：

固定信道的红色散点在每个 SNR 处从 0 到 1 均有分布，瑞利衰落信道系数的好坏直接决定全部 100 帧的传输情况，呈现极端分化。跳频蓝线始终平滑，每帧独立信道使得深衰落只影响个别帧，不会使得全部帧同时失败，FER 稳定可预测。跳频分集增益的本质不在于降低平均 FER，而在于消除落在深衰落区域导致的全部帧传输失败的极端风险，用少量平均性能换取链路可靠性的确定性。


---

## 课后实践

请补全下方跳频分集仿真中的 **3 处空缺**（每处一行代码），在 SNR=6 dB 下对比跳频与固定信道在 Rayleigh 衰落下的 FER。

要求：

1. 补全跳频信道系数 h 的生成
2. 补全固定信道接收信号的计算
3. 补全固定信道 LLR 的计算

完成后运行 ，观察跳频与固定信道的 FER 差异。

In [ ]:
%%writefile hopping_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder
from nearlink_sdr.phy.channel import ChannelConfig, ChannelModel

snr_db = 6.0
n_frames = 100
code_n, K = 256, 112
snr_lin = 10.0 ** (snr_db / 10.0)

rng = np.random.default_rng(42)
enc = PolarEncoder(code_n, K)
dec = get_polar_decoder(code_n, K)

# 生成固定信道的 h（所有帧共用）
ch_fixed = ChannelModel(config=ChannelConfig(
    snr_db=snr_db, channel_type="rayleigh",
    seed=int(rng.integers(0, 2**31))))
h_fixed = ch_fixed.get_channel_taps(code_n)[0, :]

err_hop = 0
err_fix = 0

for _ in range(n_frames):
    info = rng.integers(0, 2, size=K, dtype=np.int8)
    coded = enc.encode(info)
    tx = (1 - 2 * coded.astype(np.float64)).astype(complex)
    noise_std = np.sqrt(1.0 / (2.0 * snr_lin))
    n = noise_std * (rng.standard_normal(code_n) + 1j * rng.standard_normal(code_n))

    # ---- 跳频：每帧独立 h ----
    ch = ChannelModel(config=ChannelConfig(
        snr_db=snr_db, channel_type="rayleigh",
        seed=int(rng.integers(0, 2**31))))
    # 1: 获取跳频信道系数  （补全）

    rx_hop = tx * h_hop + n
    llr_hop = 4 * np.real(rx_hop * np.conj(h_hop)) * snr_lin
    if np.sum(dec.decode(llr_hop) != info) > 0:
        err_hop += 1

    # ---- 固定信道：所有帧共用 h_fixed ----
    #2: 固定信道接收信号  （补全）

    #3: 计算固定信道 LLR  （补全）

    if np.sum(dec.decode(llr_fix) != info) > 0:
        err_fix += 1

fer_hop = err_hop / n_frames
fer_fix = err_fix / n_frames
print(f"SNR={snr_db:.0f} dB | Hopping FER={fer_hop:.4f} | Fixed FER={fer_fix:.4f}")


执行以下命令进行编译并验证结果：


In [ ]:
!python hopping_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/05.03_answer.txt
